# Monte Carlo Methods: Off-Policy Importance Sampling

This notebook explores **Off-Policy Monte Carlo** methods. Unlike on-policy methods where the agent learns about the policy it is currently following, off-policy methods allow an agent to learn about a **Target Policy** ($\pi$) while following a different **Behavior Policy** ($b$).

## 1. Conceptual Overview: On-Policy vs. Off-Policy

In RL, we often face a dilemma: we want to learn the optimal (greedy) policy, but we need to keep exploring to find it. 

- **On-Policy**: Learns about the policy being used to make decisions (e.g., $\epsilon$-greedy). It is "compromised" because it must include exploration.
- **Off-Policy**: Learns about a "Target" policy (usually the pure greedy one) while using a separate "Behavior" policy to collect data. This allows us to learn the optimal path even while acting randomly.

### The Importance Sampling Mechanism

To learn from data generated by a different policy, we use **Importance Sampling**. We weight the returns by the relative probability of the actions being taken under the target policy versus the behavior policy.

<svg width="800" height="300" viewBox="0 0 800 300" xmlns="http://www.w3.org/2000/svg">
  <style>
    .box { fill: #ffffff; stroke: #343a40; stroke-width: 2; }
    .title { font-family: sans-serif; font-size: 14px; font-weight: bold; fill: #212529; }
    .desc { font-family: sans-serif; font-size: 11px; fill: #495057; }
    .arrow { fill: none; stroke: #007bff; stroke-width: 2; marker-end: url(#arrowhead); }
    .highlight-b { fill: #fff4e6; stroke: #fd7e14; }
    .highlight-t { fill: #e7f5ff; stroke: #007bff; }
    .label { font-family: sans-serif; font-size: 12px; fill: #6c757d; font-weight: bold; }
  </style>
  <defs>
    <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="0" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#007bff" />
    </marker>
  </defs>
  <!-- Behavior Policy Box -->
  <rect x="50" y="100" width="150" height="80" class="box highlight-b" rx="8" />
  <text x="125" y="130" class="title" text-anchor="middle">Behavior Policy (b)</text>
  <text x="125" y="155" class="desc" text-anchor="middle">Explores the World</text>
  <!-- Episode Box -->
  <rect x="250" y="100" width="120" height="80" class="box" rx="8" />
  <text x="310" y="145" class="title" text-anchor="middle">Episode Data</text>
  <!-- IS Box -->
  <rect x="420" y="100" width="180" height="80" class="box" rx="8" />
  <text x="510" y="130" class="title" text-anchor="middle">Importance Sampling</text>
  <text x="510" y="155" class="desc" text-anchor="middle">Weight: π(A|S) / b(A|S)</text>
  <!-- Target Policy Box -->
  <rect x="650" y="100" width="130" height="80" class="box highlight-t" rx="8" />
  <text x="715" y="130" class="title" text-anchor="middle">Target Policy (π)</text>
  <text x="715" y="155" class="desc" text-anchor="middle">Learns Optimally</text>
  <!-- Arrows -->
  <path d="M 200 140 L 235 140" class="arrow" />
  <path d="M 370 140 L 405 140" class="arrow" />
  <path d="M 600 140 L 635 140" class="arrow" />
  <!-- Loop Back -->
  <path d="M 715 180 L 715 240 L 510 240 L 510 180" class="arrow" />
  <text x="612" y="230" class="label" text-anchor="middle">Refines Estimates (Q)</text>
</svg>

## 2. Mathematical Foundations

### Importance Sampling Ratio ($\rho$)
The probability of the observed trajectory under policy $\pi$ divided by its probability under policy $b$:
$$\rho_{t:T-1} \doteq \prod_{k=t}^{T-1} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

### Weighted Importance Sampling Update
We use a cumulative weight $C(s, a)$ to calculate the weighted average of returns:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \frac{W}{C(S_t, A_t)} [G - Q(S_t, A_t)]$$
Where $W$ is the importance sampling ratio.

## 3. Environment: 3x3 GridWorld

We reuse the `SimpleGridWorld` from the previous pedagogy notebook. It is a 3x3 grid where State 8 is the goal (Reward +10) and all other moves cost -1.

In [ ]:
import numpy as np
import random
from collections import defaultdict
import matplotlib.pyplot as plt

class SimpleGridWorld:
    def __init__(self):
        self.size = 3
        self.goal = 8
    
    def reset(self, start_state=0):
        self.state = start_state
        return self.state
    
    def step(self, action):
        row, col = divmod(self.state, self.size)
        if action == 0: row = max(0, row - 1)    # Up
        elif action == 1: row = min(self.size - 1, row + 1) # Down
        elif action == 2: col = max(0, col - 1)  # Left
        elif action == 3: col = min(self.size - 1, col + 1) # Right
        
        self.state = row * self.size + col
        reward = 10 if self.state == self.goal else -1
        done = (self.state == self.goal)
        return self.state, reward, done

## 4. Modular Implementation

We break the Off-Policy MC algorithm into distinct functions to clarify the role of the Behavior Policy and the Importance Sampling weight updates.

In [ ]:
def get_behavior_policy_action(state, Q, epsilon=0.3):
    """
    Behavior policy is typically epsilon-greedy or uniform random to ensure coverage.
    """
    if random.random() < epsilon:
        return random.choice(range(4))
    return np.argmax(Q[state])

def generate_episode(env, Q, behavior_eps=0.3):
    """
    Generates an episode following the BEHAVIOR policy.
    """
    episode = []
    state = env.reset(0)
    done = False
    while not done:
        action = get_behavior_policy_action(state, Q, behavior_eps)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if len(episode) > 100: break # Safety cutoff
    return episode

def update_off_policy_q(episode, Q, C, target_policy, gamma=0.9):
    """
    Updates Q-values using Weighted Importance Sampling.
    """
    G = 0.0
    W = 1.0 # Importance sampling ratio
    
    # Iterate backwards through the episode
    for t in range(len(episode) - 1, -1, -1):
        s, a, r = episode[t]
        G = gamma * G + r
        
        # Update cumulative weight and Q-value
        C[s][a] += W
        Q[s][a] += (W / C[s][a]) * (G - Q[s][a])
        
        # Update target policy (greedy)
        target_policy[s] = np.argmax(Q[s])
        
        # If the action taken by behavior policy is NOT what target policy would do,
        # the probability of this tail under target policy is 0. We can stop.
        if a != target_policy[s]:
            break
            
        # Update W: W *= 1 / Pr(A|S) under behavior policy
        # Assuming behavior policy is uniform random for simplicity here (Pr = 0.25)
        # Or more accurately, use the actual probability from behavior policy
        # In this demo, we'll assume a fixed exploration behavior probability
        W = W * (1.0 / 0.25) # Simplified for uniform behavior

## 5. Running the Off-Policy Control

We will now run the agent. Note how the `target_policy` is updated even though the `episode` is generated using a noisy `behavior_policy`.

In [ ]:
def run_off_policy_mc(num_episodes=500):
    env = SimpleGridWorld()
    Q = defaultdict(lambda: np.zeros(4))
    C = defaultdict(lambda: np.zeros(4))
    target_policy = defaultdict(lambda: random.choice(range(4)))
    
    print(f"--- Running Off-Policy MC Control ({num_episodes} episodes) ---")
    
    for i in range(num_episodes):
        # Generate episode using behavior policy (Uniform Random for high coverage)
        episode = []
        state = env.reset(0)
        done = False
        while not done:
            action = random.choice(range(4)) # Purely random behavior
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if len(episode) > 50: break
            
        update_off_policy_q(episode, Q, C, target_policy)
        
        if (i+1) % 100 == 0:
            print(f"Episode {i+1} completed...")

    return target_policy, Q

target_policy, Q = run_off_policy_mc()

print("\nFinal Target Policy (Greedy) Grid:")
policy_grid = np.array([target_policy[s] for s in range(9)]).reshape(3,3)
print(policy_grid)
print("\n(0: Up, 1: Down, 2: Left, 3: Right)")

## 6. Key Takeaways

1. **Decoupling**: Off-policy methods decouple exploration (Behavior) from exploitation (Target). 
2. **Coverage**: The behavior policy must have a non-zero probability of taking any action that the target policy might take (Assumption of Coverage).
3. **Efficiency**: While powerful, importance sampling can have high variance, especially with long episodes. Weighted importance sampling helps mitigate this by reducing the variance of the estimates.